In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from src.utils.config_loader import ConfigLoader

# Initialize config
cfg = ConfigLoader("../config/config.yml")

# Sanity checks
print("Catalog:", cfg.catalog)
print("Bronze Transactions Table:", cfg.get_full_table_name("bronze", "transactions"))
print("Transactions Source Path:", cfg.get_raw_csv_path("transactions_path"))
print("Checkpoint Path:", cfg.get_checkpoint_path("autoloader_transactions"))

In [0]:
from src.ingestion.batch_ingestion import run_batch_csv_ingestion
from src.utils.config_loader import ConfigLoader

cfg = ConfigLoader("../config/config.yml")
run_batch_csv_ingestion(spark, cfg)

In [0]:
%sql
-- Check total ingested row counts across all batch tables
SELECT 'transactions' AS table_name, COUNT(*) AS row_count FROM maven_market_uc.bronze.transactions
UNION ALL
SELECT 'stores', COUNT(*) FROM maven_market_uc.bronze.stores
UNION ALL
SELECT 'regions', COUNT(*) FROM maven_market_uc.bronze.regions
UNION ALL
SELECT 'calendar', COUNT(*) FROM maven_market_uc.bronze.calendar
UNION ALL
SELECT 'returns', COUNT(*) FROM maven_market_uc.bronze.returns;

In [0]:
%sql
-- Verify ingestion metadata fields
SELECT _source_file_name, _ingested_at, COUNT(*) AS file_record_count 
FROM maven_market_uc.bronze.transactions 
GROUP BY _source_file_name, _ingested_at;

In [0]:
# Command 1 (Top cell in your notebook)
%pip install pymongo

In [0]:
from src.ingestion.mongodb_ingestion import run_mongodb_ingestion
from src.utils.config_loader import ConfigLoader

cfg = ConfigLoader("../config/config.yml")
run_mongodb_ingestion(spark, cfg, secret_scope="scope-maven-market")

In [0]:
%sql
SELECT 'mongodb_products' AS source_table, COUNT(*) AS row_count, MAX(_ingested_at) AS latest_ingested FROM maven_market_uc.bronze.mongodb_products
UNION ALL
SELECT 'mongodb_customers', COUNT(*), MAX(_ingested_at) FROM maven_market_uc.bronze.mongodb_customers;

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.get(scope="scope-maven-market", key="mongodb-connection-uri")

In [0]:
scope = "scope-maven-market"
print("Bootstrap Server:", dbutils.secrets.get(scope=scope, key="kafka-bootstrap-servers"))
print("API Key Exists:", len(dbutils.secrets.get(scope=scope, key="kafka-api-key")) > 0)
print("API Secret Exists:", len(dbutils.secrets.get(scope=scope, key="kafka-api-secret")) > 0)

In [0]:
# Databricks Notebook: 01_run_kafka_producer
from src.generators.kafka_producer import run_kafka_producer
from src.utils.config_loader import ConfigLoader

# 1. Load configuration
cfg = ConfigLoader("../config/config.yml")

# 2. Run producer to stream 300 event pairs (orders & inventory) into Kafka
# Adjustment: delay=0.3s means it will publish for ~1.5 to 2 minutes
run_kafka_producer(spark, cfg, secret_scope="scope-maven-market", num_events=300, delay=0.3)

In [0]:
# Databricks Notebook: 02_run_kafka_consumer
from src.ingestion.kafka_stream import run_kafka_streaming_consumer
from src.utils.config_loader import ConfigLoader

# 1. Load configuration
cfg = ConfigLoader("../config/config.yml")

# 2. Launch Spark Structured Streaming pipelines
queries = run_kafka_streaming_consumer(spark, cfg, secret_scope="scope-maven-market")

# The streaming query dashboard will appear inline below the cell!

In [0]:
from databricks.sdk.runtime import dbutils
from confluent_kafka import Consumer

scope = "scope-maven-market"

# 1. Fetch & sanitize secret values
bootstrap = dbutils.secrets.get(scope=scope, key="kafka-bootstrap-servers").strip()
api_key = dbutils.secrets.get(scope=scope, key="kafka-api-key").strip()
api_secret = dbutils.secrets.get(scope=scope, key="kafka-api-secret").strip()

print(f"Bootstrap Server: '{bootstrap}'")
print(f"API Key Length: {len(api_key)}")
print(f"API Secret Length: {len(api_secret)}")

# 2. Test raw Confluent Cloud connection
conf = {
    'bootstrap.servers': bootstrap,
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': api_key,
    'sasl.password': api_secret,
    'group.id': 'diagnostic-test-group'
}

try:
    c = Consumer(conf)
    metadata = c.list_topics(timeout=10)
    print(" Successfully connected to Confluent Cloud!")
    print(" Available Cluster Topics:", list(metadata.topics.keys()))
    c.close()
except Exception as e:
    print(" Connection Test Failed:", e)

In [0]:
from databricks.sdk.runtime import dbutils
from confluent_kafka import Consumer

scope = "scope-maven-market"

# Fetch and sanitize secret values
bootstrap = dbutils.secrets.get(scope=scope, key="kafka-bootstrap-servers").strip().replace("https://", "").replace("http://", "").rstrip("/")
api_key = dbutils.secrets.get(scope=scope, key="kafka-api-key").strip()
api_secret = dbutils.secrets.get(scope=scope, key="kafka-api-secret").strip()

print(f"Sanitized Bootstrap Server: '{bootstrap}'")
print(f"API Key Length: {len(api_key)}")
print(f"API Secret Length: {len(api_secret)}")

conf = {
    'bootstrap.servers': bootstrap,
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': api_key,
    'sasl.password': api_secret,
    'group.id': 'databricks-test-group'
}

c = Consumer(conf)
metadata = c.list_topics(timeout=10)
print("Confluent Cloud Connection Successful!")
print("Available Topics:", list(metadata.topics.keys()))
c.close()

In [0]:
import importlib
import src.ingestion.kafka_stream as ks
from src.utils.config_loader import ConfigLoader

# Force Python to reload module changes from workspace storage
importlib.reload(ks)

cfg = ConfigLoader("../config/config.yml")
queries = ks.run_kafka_streaming_consumer(spark, cfg, secret_scope="scope-maven-market")

In [0]:
%sql
SELECT * FROM maven_market_uc.bronze.mongodb_customers
LIMIT 10;


In [0]:
%sh
databricks secrets list-keys --scope scope-maven-market

In [0]:
dbutils.secrets.get(scope="scope-maven-market", key="mongodb-connection-uri")

In [0]:
dbutils.secrets.listScopes()

In [0]:

import json
import pymongo
import pandas as pd
from pyspark.sql.functions import current_timestamp, col, input_file_name, lit 
from src.utils.dlt_loader import load_config
from databricks.sdk.runtime import dbutils

cfg = load_config()
base_path = cfg["storage"]["base_path"]
secret_scope = cfg["secret_scope"]
catalog = cfg["catalog"]
bronze_schema = cfg["targets"]["schemas"]["bronze"]

mongo_cfg = cfg["sources"]["mongodb"]
db_name = mongo_cfg["database"]
mongo_secret_key = mongo_cfg.get("uri_secret_key", "mongodb-connection-uri")

# Retrieve MongoDB connection string securely from Azure Key Vault
mongo_uri = dbutils.secrets.get(scope=secret_scope, key=mongo_secret_key).strip()

for entity_name, collection_name in mongo_cfg["collections"].items():
    dlt_tbl_name = f"bronze_mongodb_{entity_name}"

    def create_mongo_dlt_table(coll_name, tbl):
        @dlt.table(
            name=tbl,  # Registers 'bronze_mongodb_customers' & 'bronze_mongodb_products'
            comment=f"Raw MongoDB Atlas direct ingestion for {coll_name} via PyMongo",
            table_properties={"quality": "bronze"}
        )
        def ingest_mongo():
            # 1. Fetch raw documents from MongoDB Atlas via PyMongo
            client = pymongo.MongoClient(mongo_uri)
            db = client[db_name]
            collection = db[coll_name]
            
            raw_docs = list(collection.find())
            print(raw_docs)

            if not raw_docs:
                # Return empty DataFrame with dummy schema if collection is empty
                pass

            # 2. Convert BSON objects into standard JSON strings
            cleaned_docs = [json.loads(json.dumps(doc, default=str)) for doc in raw_docs]

            # 3. Convert to Pandas DataFrame first to unify numeric types
            df_pd = pd.DataFrame(cleaned_docs)

            # 4. Convert Pandas DataFrame to PySpark DataFrame
            df_mongo = spark.createDataFrame(df_pd)

            # 5. Enrich metadata and return DataFrame (DLT handles saving to Unity Catalog)
            return (
                df_mongo
                .withColumn("_ingested_at", current_timestamp())
                .withColumn("_source_system", lit(f"MongoDB_Atlas.{coll_name}"))
            )

    create_mongo_dlt_table(collection_name, dlt_tbl_name)

In [0]:
%sql
SELECT COUNT(*) AS total_raw_records 
FROM maven_market_uc.bronze.bronze_kafka_inventory;

In [0]:
%sql
SELECT CAST(value AS STRING) AS raw_payload 
FROM maven_market_uc.bronze.bronze_kafka_orders 
LIMIT 5;